# Oxford → tabla `avance_oxford`

Validación paso a paso. Corre las celdas **en orden**: cada una dice cuánto tardó y qué salió,
así se ve enseguida en cuál se traba.

| Celda | Qué comprueba |
|---|---|
| 1 | Que el kernel responde y que la ruta existe (no usa pandas) |
| 2 | Que pandas carga |
| 3-4 | Que el archivo se abre y cuánto pesa cada hoja |
| 5-6 | Que se reconoce el encabezado de dos filas |
| 7-8 | La transformación de ancho a largo |
| 9-10 | Revisión del dataset y comparación contra el archivo original |
| 11 | Guardar el resultado |

In [1]:
# 1. ¿El kernel responde? Si esta celda no termina, el problema es el kernel, no el codigo.
import time
from pathlib import Path

ORIGEN = Path.cwd() / "Descargas" / "Oxford"    # la carpeta entera (o un archivo suelto)
GUARDAR_EN = Path.cwd() / "avance_oxford.csv"   # respaldo de lo que se cargo
QUITAR_PROMEDIO = True          # la fila "Class Average" no es un estudiante

print("kernel vivo:", time.strftime("%H:%M:%S"))
print("origen:", ORIGEN)
print("existe:", ORIGEN.exists(), "| es carpeta:", ORIGEN.is_dir())

kernel vivo: 15:50:01
origen: C:\Users\sebastian.mendez\OneDrive - CENTRO COLOMBO AMERICANO\Escritorio\Automatizacion Oxford\Descargas\Oxford
existe: True | es carpeta: True


In [2]:
%pip install sqlalchemy pyodbc

Note: you may need to restart the kernel to use updated packages.


In [3]:
# 2. Cargar pandas (la primera vez puede tardar unos segundos)
inicio = time.time()
import re

import pandas as pd

print(f"pandas {pd.__version__} listo en {time.time() - inicio:.1f} s")

pandas 3.0.6 listo en 3.6 s


In [4]:
# 3. Que archivos y hojas vamos a leer (todos los de la carpeta)
EXTENSIONES = (".csv", ".xlsx", ".xls", ".xlsm")   # lo demas que haya en la carpeta se ignora


def hojas_de(ruta):
    """Nombres de las hojas de un Excel; para un CSV, una sola 'hoja' sin nombre."""
    if ruta.suffix.lower() in (".xlsx", ".xls", ".xlsm"):
        return pd.ExcelFile(ruta).sheet_names
    return [None]


archivos = ([a for a in sorted(ORIGEN.iterdir()) if a.suffix.lower() in EXTENSIONES]
            if ORIGEN.is_dir() else [ORIGEN])
inicio = time.time()
agenda = [(archivo, hoja) for archivo in archivos for hoja in hojas_de(archivo)]
print(f"{len(archivos)} archivo(s), {len(agenda)} hoja(s) en {time.time() - inicio:.1f} s")
for archivo, hoja in agenda:
    print(f"  {archivo.name}" + (f"  |  {hoja}" if hoja else ""))

10 archivo(s), 10 hoja(s) en 0.0 s
  2026-09-22_1539_B1-2-5PM24_GT.csv
  2026-09-22_1539_B1.2-5PM112_GT.csv
  2026-09-22_1540_B1.2-5PM113_GT.csv
  2026-09-22_1540_B1.2-5PM114_IAV.csv
  2026-09-22_1540_B1.2-5PM115_IAV.csv
  2026-09-22_1541_B1.2-5PM22_BPO.csv
  2026-09-22_1541_B1.2-5PM23_BPO.csv
  2026-09-22_1541_B1.2-5PM26_TIC.csv
  2026-09-22_1542_B1.2-5PM27_TIC.csv
  2026-09-22_1542_B1.2-5PM28_IAV.csv


In [5]:
# 4. Leer cada hoja TAL CUAL: sin encabezados y todo como texto
def codificacion_de(ruta):
    """Adivina la codificacion por la firma (BOM) del archivo. Oxford exporta en UTF-16."""
    firma = ruta.read_bytes()[:3]
    if firma.startswith((b"\xff\xfe", b"\xfe\xff")):
        return "utf-16"
    if firma.startswith(b"\xef\xbb\xbf"):
        return "utf-8-sig"
    return "utf-8"


def separador_de(ruta, juego):
    """El separador mas repetido en la primera linea (Oxford usa tabulador, no coma)."""
    with open(ruta, encoding=juego, errors="replace", newline="") as archivo:
        primera = archivo.readline()
    return max(("\t", ",", ";", "|"), key=primera.count)


def leer_crudo(ruta, hoja=None):
    """Devuelve la hoja tal como esta en el archivo, sin interpretar encabezados."""
    if ruta.suffix.lower() in (".xlsx", ".xls", ".xlsm"):
        return pd.read_excel(ruta, sheet_name=hoja, header=None, dtype=object)
    juego = codificacion_de(ruta)
    return pd.read_csv(ruta, header=None, dtype=object, encoding=juego,
                       sep=separador_de(ruta, juego), engine="python")


crudos = {}
for archivo, hoja in agenda:
    inicio = time.time()
    crudo = leer_crudo(archivo, hoja)
    crudos[(archivo.name, hoja)] = crudo
    detalle = "" if hoja else f"  [{codificacion_de(archivo)}]"
    print(f"  {archivo.name} | {hoja}: {crudo.shape[0]} filas x {crudo.shape[1]} columnas"
          f"{detalle}  ({time.time() - inicio:.1f} s)")


  2026-09-22_1539_B1-2-5PM24_GT.csv | None: 21 filas x 54 columnas  [utf-16]  (0.0 s)
  2026-09-22_1539_B1.2-5PM112_GT.csv | None: 4 filas x 54 columnas  [utf-16]  (0.0 s)
  2026-09-22_1540_B1.2-5PM113_GT.csv | None: 4 filas x 54 columnas  [utf-16]  (0.0 s)
  2026-09-22_1540_B1.2-5PM114_IAV.csv | None: 4 filas x 54 columnas  [utf-16]  (0.0 s)
  2026-09-22_1540_B1.2-5PM115_IAV.csv | None: 4 filas x 54 columnas  [utf-16]  (0.0 s)
  2026-09-22_1541_B1.2-5PM22_BPO.csv | None: 26 filas x 54 columnas  [utf-16]  (0.0 s)
  2026-09-22_1541_B1.2-5PM23_BPO.csv | None: 26 filas x 54 columnas  [utf-16]  (0.0 s)
  2026-09-22_1541_B1.2-5PM26_TIC.csv | None: 31 filas x 54 columnas  [utf-16]  (0.0 s)
  2026-09-22_1542_B1.2-5PM27_TIC.csv | None: 34 filas x 54 columnas  [utf-16]  (0.0 s)
  2026-09-22_1542_B1.2-5PM28_IAV.csv | None: 17 filas x 54 columnas  [utf-16]  (0.0 s)


In [6]:
# 5. Las funciones de apoyo
def texto(serie):
    """Serie de texto limpio y sin NaN, para comparar encabezados sin sorpresas."""
    return serie.astype("string").fillna("").str.strip()


def a_porcentaje(serie):
    """'42.9%' -> 42.9   |   '-', 'Never', vacio -> NaN"""
    return pd.to_numeric(texto(serie).str.replace("%", "", regex=False),
                         errors="coerce").round(2)


def a_fecha(serie):
    """'03 Dec 2025' -> 2025-12-03   |   'Never' y '-' -> NaT"""
    return pd.to_datetime(texto(serie), format="%d %b %Y", errors="coerce")


def partir_bloque(titulo):
    """'Unit 1 Achievements' -> (1, 'Achievements', 'Practice')
       'Unit 1 Test'         -> (1, 'Test', 'Test')
       'Total'               -> (NA, 'Total / General', 'Overall')"""
    titulo = " ".join(str(titulo).split())
    if titulo.lower().startswith("total"):
        return pd.NA, "Total / General", "Overall"
    partes = re.match(r"(?i)^unit\s+(\d+)\s*(.*)$", titulo)
    if not partes:
        return pd.NA, titulo, "Practice"
    resto = partes[2].strip() or "Unit"
    return int(partes[1]), resto, ("Test" if resto.lower().endswith("test") else "Practice")


def fila_encabezado(crudo, buscar_en=20):
    """Numero de la fila que dice 'Completed' (la segunda del encabezado), o None."""
    arriba = crudo.head(buscar_en)
    marca = arriba.apply(lambda fila: texto(fila).str.lower().eq("completed").any(), axis=1)
    return int(marca.idxmax()) if marca.any() else None


print("funciones listas")
print("  '42.9%' ->", a_porcentaje(pd.Series(["42.9%", "-", "0%", "Never"])).tolist())
print("  fechas  ->", a_fecha(pd.Series(["03 Dec 2025", "Never", "-"])).dt.date.tolist())
for ejemplo in ("Total", "Unit 1 Achievements", "Unit 12 Test", "Unit 5 Progress Test"):
    print(f"  {ejemplo:<22} -> {partir_bloque(ejemplo)}")

funciones listas
  '42.9%' -> [42.9, <NA>, 0.0, <NA>]
  fechas  -> [datetime.date(2025, 12, 3), NaT, NaT]
  Total                  -> (<NA>, 'Total / General', 'Overall')
  Unit 1 Achievements    -> (1, 'Achievements', 'Practice')
  Unit 12 Test           -> (12, 'Test', 'Test')
  Unit 5 Progress Test   -> (5, 'Progress Test', 'Test')


In [7]:
# 6. Ubicar el encabezado de cada hoja y mirarlo con los propios ojos
for clave, crudo in crudos.items():
    fila_sub = fila_encabezado(crudo) if len(crudo) else None
    print(f"{clave}: " + (f"encabezado en las filas {fila_sub - 1} y {fila_sub}"
                          if fila_sub else "no es del Progress Tracker, se salta"))

clave_gt = next(c for c, cr in crudos.items() if len(cr) and fila_encabezado(cr))
crudo_gt = crudos[clave_gt]
fila_sub = fila_encabezado(crudo_gt)
print(f"\nHoja de ejemplo: {clave_gt}")
crudo_gt.iloc[fila_sub - 1:fila_sub + 3, :10]

('2026-09-22_1539_B1-2-5PM24_GT.csv', None): encabezado en las filas 0 y 1
('2026-09-22_1539_B1.2-5PM112_GT.csv', None): encabezado en las filas 0 y 1
('2026-09-22_1540_B1.2-5PM113_GT.csv', None): encabezado en las filas 0 y 1
('2026-09-22_1540_B1.2-5PM114_IAV.csv', None): encabezado en las filas 0 y 1
('2026-09-22_1540_B1.2-5PM115_IAV.csv', None): encabezado en las filas 0 y 1
('2026-09-22_1541_B1.2-5PM22_BPO.csv', None): encabezado en las filas 0 y 1
('2026-09-22_1541_B1.2-5PM23_BPO.csv', None): encabezado en las filas 0 y 1
('2026-09-22_1541_B1.2-5PM26_TIC.csv', None): encabezado en las filas 0 y 1
('2026-09-22_1542_B1.2-5PM27_TIC.csv', None): encabezado en las filas 0 y 1


('2026-09-22_1542_B1.2-5PM28_IAV.csv', None): encabezado en las filas 0 y 1



Hoja de ejemplo: ('2026-09-22_1539_B1-2-5PM24_GT.csv', None)


,0,1,2,3,4,5,6,7,8,9
0,NaN,NaN,NaN,Last Opened,Total,NaN,Unit 1 Achievements,NaN,Unit 1 Test,NaN
1,Teacher,Learning Material,Student,Date,Completed,Score,Completed,Score,Completed,Score
2,Daniel Alvarez,Wide Angle Level 4 Online Practice,Class Average,-,42.9%,79.3%,50.0%,91.1%,50.0%,100.0%
3,NaN,NaN,Armando Pacheco,03 Dec 2025,0.0%,-,0.0%,-,0.0%,-


In [8]:
# 7. De ancho a largo: una fila por estudiante y bloque
def a_largo(crudo, origen=""):
    """Convierte una hoja ancha del Progress Tracker en filas."""
    fila_sub = fila_encabezado(crudo)
    if not fila_sub:                      # sin fila 'Completed' (o esta en la primera fila)
        return None

    subtitulos = texto(crudo.loc[fila_sub])            # Teacher, Student, Completed, Score...
    # El nombre del bloque solo esta encima de "Completed"; lo de al lado se arrastra
    titulos = (texto(crudo.loc[fila_sub - 1])
               .replace(r"^_\d+$", pd.NA, regex=True)  # Excel pone _1, _2... en las vacias
               .replace("", pd.NA).ffill().fillna(""))
    datos = crudo.loc[fila_sub + 1:].reset_index(drop=True)

    def columna(nombre):
        """Numero de la columna cuyo subtitulo es ese (o None si no esta)."""
        halladas = subtitulos.index[subtitulos.str.lower() == nombre]
        return halladas[0] if len(halladas) else None

    c_docente, c_material = columna("teacher"), columna("learning material")
    c_estudiante, c_fecha = columna("student"), columna("date")

    # Docente y material vienen solo en la fila de "Class Average": se arrastran hacia abajo
    fijas = pd.DataFrame({
        "teacher": (texto(datos[c_docente]).replace("", pd.NA).ffill()
                    if c_docente is not None else pd.NA),
        "learning_material": (texto(datos[c_material]).replace("", pd.NA).ffill()
                              if c_material is not None else pd.NA),
        "student": texto(datos[c_estudiante]) if c_estudiante is not None else pd.NA,
        "last_opened": a_fecha(datos[c_fecha]) if c_fecha is not None else pd.NaT,
    })

    # Un pedazo por cada par Completed / Score
    piezas = []
    for orden, inicio in enumerate(subtitulos.index[subtitulos.str.lower() == "completed"]):
        numero, unidad, tipo = partir_bloque(titulos[inicio])
        siguiente = inicio + 1
        hay_score = siguiente in subtitulos.index and subtitulos[siguiente].lower() == "score"
        piezas.append(fijas.assign(
            unit_number=numero,
            unit_title=unidad,
            item_type=tipo,
            completed_pct=a_porcentaje(datos[inicio]),
            score_pct=a_porcentaje(datos[siguiente]) if hay_score else pd.NA,
            _fila=range(len(datos)),      # para dejar juntas las filas de cada estudiante
            _orden=orden,
        ))

    largo = pd.concat(piezas, ignore_index=True)
    largo = largo.sort_values(["_fila", "_orden"]).drop(columns=["_fila", "_orden"])
    largo = largo[largo["student"].ne("")]
    if QUITAR_PROMEDIO:
        largo = largo[largo["student"].str.lower() != "class average"]
    largo["unit_number"] = largo["unit_number"].astype("Int64")
    largo["origen"] = origen
    return largo.reset_index(drop=True)


inicio = time.time()
prueba = a_largo(crudo_gt, str(clave_gt))
print(f"{len(prueba)} filas en {time.time() - inicio:.1f} s")
prueba.head(10)

450 filas en 0.0 s


,teacher,learning_material,student,last_opened,unit_number,unit_title,item_type,completed_pct,score_pct,origen
0,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,<NA>,Total / General,Overall,0.0,<NA>,"('2026-09-22_1539_B1-2-5PM24_GT.csv', None)"
1,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,1,Achievements,Practice,0.0,<NA>,"('2026-09-22_1539_B1-2-5PM24_GT.csv', None)"
2,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,1,Test,Test,0.0,<NA>,"('2026-09-22_1539_B1-2-5PM24_GT.csv', None)"
3,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,2,News,Practice,0.0,<NA>,"('2026-09-22_1539_B1-2-5PM24_GT.csv', None)"
4,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,2,Test,Test,0.0,<NA>,"('2026-09-22_1539_B1-2-5PM24_GT.csv', None)"
5,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,3,Frontiers,Practice,0.0,<NA>,"('2026-09-22_1539_B1-2-5PM24_GT.csv', None)"
6,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,3,Test,Test,0.0,<NA>,"('2026-09-22_1539_B1-2-5PM24_GT.csv', None)"
7,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,4,Processes,Practice,0.0,<NA>,"('2026-09-22_1539_B1-2-5PM24_GT.csv', None)"
8,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,4,Test,Test,0.0,<NA>,"('2026-09-22_1539_B1-2-5PM24_GT.csv', None)"
9,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,5,Survival,Practice,0.0,<NA>,"('2026-09-22_1539_B1-2-5PM24_GT.csv', None)"


In [9]:
# 8. Todas las hojas de todos los archivos
partes = []
for clave, crudo in crudos.items():
    if not len(crudo):
        continue
    largo = a_largo(crudo, f"{clave[0]} | {clave[1]}" if clave[1] else clave[0])
    print(f"  {clave}: " + (f"{len(largo)} filas" if largo is not None else "se salta"))
    if largo is not None:
        partes.append(largo)

avance = pd.concat(partes, ignore_index=True)
print(f"\nTotal: {len(avance):,} filas")
avance.head()

  ('2026-09-22_1539_B1-2-5PM24_GT.csv', None): 450 filas


  ('2026-09-22_1539_B1.2-5PM112_GT.csv', None): 0 filas
  ('2026-09-22_1540_B1.2-5PM113_GT.csv', None): 0 filas


  ('2026-09-22_1540_B1.2-5PM114_IAV.csv', None): 0 filas
  ('2026-09-22_1540_B1.2-5PM115_IAV.csv', None): 0 filas


  ('2026-09-22_1541_B1.2-5PM22_BPO.csv', None): 575 filas


  ('2026-09-22_1541_B1.2-5PM23_BPO.csv', None): 575 filas


  ('2026-09-22_1541_B1.2-5PM26_TIC.csv', None): 700 filas
  ('2026-09-22_1542_B1.2-5PM27_TIC.csv', None): 775 filas


  ('2026-09-22_1542_B1.2-5PM28_IAV.csv', None): 350 filas

Total: 3,425 filas


,teacher,learning_material,student,last_opened,unit_number,unit_title,item_type,completed_pct,score_pct,origen
0,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,<NA>,Total / General,Overall,0.0,<NA>,2026-09-22_1539_B1-2-5PM24_GT.csv
1,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,1,Achievements,Practice,0.0,<NA>,2026-09-22_1539_B1-2-5PM24_GT.csv
2,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,1,Test,Test,0.0,<NA>,2026-09-22_1539_B1-2-5PM24_GT.csv
3,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,2,News,Practice,0.0,<NA>,2026-09-22_1539_B1-2-5PM24_GT.csv
4,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,2,Test,Test,0.0,<NA>,2026-09-22_1539_B1-2-5PM24_GT.csv


In [10]:
# 9. Revision del dataset
print(avance.dtypes, "\n")
print("estudiantes:", avance["student"].nunique(),
      "| materiales:", avance["learning_material"].nunique(),
      "| docentes:", avance["teacher"].nunique())
print("\nfilas por tipo:\n", avance["item_type"].value_counts())
print("\nvacios por columna:\n", avance.isna().sum())
print("\ncompleted_pct:", (avance["completed_pct"].min(), avance["completed_pct"].max()),
      "| score_pct:", (avance["score_pct"].min(), avance["score_pct"].max()))

# Cada estudiante debe tener tantas filas como bloques 'Completed' tenga su hoja
por_estudiante = avance.groupby(["origen", "student"]).size()
print("\nfilas por estudiante (min / max):", por_estudiante.min(), "/", por_estudiante.max())

teacher                      string
learning_material            string
student                      string
last_opened          datetime64[us]
unit_number                   Int64
unit_title                      str
item_type                       str
completed_pct               Float64
score_pct                   Float64
origen                          str
dtype: object 

estudiantes: 137 | materiales: 1 | docentes: 1

filas por tipo:
 item_type
Practice    1644
Test        1644
Overall      137
Name: count, dtype: int64

vacios por columna:
 teacher                 0
learning_material       0
student                 0
last_opened          2075
unit_number           137
unit_title              0
item_type               0
completed_pct           0
score_pct            3025
origen                  0
dtype: int64

completed_pct: (np.float64(0.0), np.float64(100.0)) | score_pct: (np.float64(0.3), np.float64(100.0))

filas por estudiante (min / max): 25 / 25


In [11]:
# 10. La prueba de fuego: una fila del archivo original contra sus filas transformadas
subtitulos = texto(crudo_gt.loc[fila_sub])
col_estudiante = subtitulos.index[subtitulos.str.lower() == "student"][0]
estudiante = prueba["student"].iloc[0]

print("Estudiante:", estudiante)
print("\n--- como viene en el archivo (primeras 12 celdas) ---")
display(crudo_gt.loc[texto(crudo_gt[col_estudiante]) == estudiante].iloc[:, :12])
print("--- como queda en el dataset (primeras 6 filas) ---")
display(prueba[prueba["student"] == estudiante].head(6))

Estudiante: Armando Pacheco

--- como viene en el archivo (primeras 12 celdas) ---


,0,1,2,3,4,5,6,7,8,9,10,11
3,NaN,NaN,Armando Pacheco,03 Dec 2025,0.0%,-,0.0%,-,0.0%,-,0.0%,-


--- como queda en el dataset (primeras 6 filas) ---


,teacher,learning_material,student,last_opened,unit_number,unit_title,item_type,completed_pct,score_pct,origen
0,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,<NA>,Total / General,Overall,0.0,<NA>,"('2026-09-22_1539_B1-2-5PM24_GT.csv', None)"
1,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,1,Achievements,Practice,0.0,<NA>,"('2026-09-22_1539_B1-2-5PM24_GT.csv', None)"
2,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,1,Test,Test,0.0,<NA>,"('2026-09-22_1539_B1-2-5PM24_GT.csv', None)"
3,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,2,News,Practice,0.0,<NA>,"('2026-09-22_1539_B1-2-5PM24_GT.csv', None)"
4,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,2,Test,Test,0.0,<NA>,"('2026-09-22_1539_B1-2-5PM24_GT.csv', None)"
5,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,3,Frontiers,Practice,0.0,<NA>,"('2026-09-22_1539_B1-2-5PM24_GT.csv', None)"


In [12]:
# 11. Dataset final: columnas de la tabla + clase/grupo/codigo/segmento del nombre del archivo

#   2026-09-22_1030_B1.2-5PM28_IAV.csv
#                   \__/ \_/\/  \_/
#                  clase grupo codigo segmento
PATRON_CURSO = re.compile(r"""(?ix)
    (?P<clase>[a-z]{1,3}\s*\d+(?:[.\-_]\d+)?)    # B1.2, B1-2, A2, C1.1
    [\s\-_]*
    (?P<grupo>\d{1,2}\s*[ap]\.?m\.?)             # 5PM, 10AM
    [\s\-_]*
    (?P<codigo>\d+)                              # 28
    (?:[\s\-_]+(?P<segmento>[a-z]{2,10}))?       # IAV, TIC, BPO, GT
""")


def partes_del_nombre(origen):
    """Saca clase, grupo, codigo y segmento del nombre del archivo (o de la hoja, si ahi esta)."""
    for trozo in str(origen).split("|"):                          # primero archivo, luego hoja
        nombre = Path(trozo.strip()).stem
        nombre = re.sub(r"^\d{4}-\d{2}-\d{2}_\d{3,4}_", "", nombre)   # fecha y hora que pone el bot
        hallado = PATRON_CURSO.search(nombre)
        if not hallado:
            continue
        return {
            "clase": re.sub(r"[\s_\-]", ".", hallado["clase"].strip()).upper(),   # B1-2 -> B1.2
            "codigo": hallado["codigo"],                          # texto: un 08 no se vuelve 8
            "grupo": re.sub(r"[\s.]", "", hallado["grupo"]).upper(),              # 5 p.m. -> 5PM
            "segmento": (hallado["segmento"] or "").upper() or None,
        }
    return {"clase": None, "codigo": None, "grupo": None, "segmento": None}


# Se resuelve una vez por archivo, no una vez por fila
nombres = pd.DataFrame([{"origen": o, **partes_del_nombre(o)}
                        for o in avance["origen"].unique()])
print("Lo que se saco de cada nombre:")
display(nombres)

avance_oxford = avance.merge(nombres, on="origen", how="left")
avance_oxford["created_at"] = pd.Timestamp.now()
avance_oxford = avance_oxford[["teacher", "learning_material", "student", "last_opened",
                               "unit_number", "unit_title", "item_type",
                               "completed_pct", "score_pct", "created_at",
                               "clase", "codigo", "grupo", "segmento"]]

sin_curso = avance_oxford["clase"].isna().sum()
if sin_curso:
    print(f"OJO: {sin_curso:,} filas sin clase (el nombre no trae el codigo del curso)")

avance_oxford.to_csv(GUARDAR_EN, index=False, encoding="utf-8-sig")
print(f"{len(avance_oxford):,} filas guardadas en {GUARDAR_EN}")
print("lote (created_at):", avance_oxford["created_at"].iloc[0])
print("para deshacer esta carga en SQL:")
print(f"    DELETE FROM dbo.avance_oxford WHERE created_at = "
      f"'{avance_oxford['created_at'].iloc[0]:%Y-%m-%d %H:%M:%S.%f}';")
avance_oxford.head(30)

Lo que se saco de cada nombre:


,origen,clase,codigo,grupo,segmento
0,2026-09-22_1539_B1-2-5PM24_GT.csv,B1.2,24,5PM,GT
1,2026-09-22_1541_B1.2-5PM22_BPO.csv,B1.2,22,5PM,BPO
2,2026-09-22_1541_B1.2-5PM23_BPO.csv,B1.2,23,5PM,BPO
3,2026-09-22_1541_B1.2-5PM26_TIC.csv,B1.2,26,5PM,TIC
4,2026-09-22_1542_B1.2-5PM27_TIC.csv,B1.2,27,5PM,TIC
5,2026-09-22_1542_B1.2-5PM28_IAV.csv,B1.2,28,5PM,IAV


3,425 filas guardadas en C:\Users\sebastian.mendez\OneDrive - CENTRO COLOMBO AMERICANO\Escritorio\Automatizacion Oxford\avance_oxford.csv
lote (created_at): 2026-09-22 15:50:08.929058
para deshacer esta carga en SQL:
    DELETE FROM dbo.avance_oxford WHERE created_at = '2026-09-22 15:50:08.929058';


,teacher,learning_material,student,last_opened,unit_number,unit_title,item_type,completed_pct,score_pct,created_at,clase,codigo,grupo,segmento
0,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,<NA>,Total / General,Overall,0.0,<NA>,2026-09-22 15:50:08.929058,B1.2,24,5PM,GT
1,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,1,Achievements,Practice,0.0,<NA>,2026-09-22 15:50:08.929058,B1.2,24,5PM,GT
2,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,1,Test,Test,0.0,<NA>,2026-09-22 15:50:08.929058,B1.2,24,5PM,GT
3,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,2,News,Practice,0.0,<NA>,2026-09-22 15:50:08.929058,B1.2,24,5PM,GT
4,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,2,Test,Test,0.0,<NA>,2026-09-22 15:50:08.929058,B1.2,24,5PM,GT
5,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,3,Frontiers,Practice,0.0,<NA>,2026-09-22 15:50:08.929058,B1.2,24,5PM,GT
6,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,3,Test,Test,0.0,<NA>,2026-09-22 15:50:08.929058,B1.2,24,5PM,GT
7,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,4,Processes,Practice,0.0,<NA>,2026-09-22 15:50:08.929058,B1.2,24,5PM,GT
8,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,4,Test,Test,0.0,<NA>,2026-09-22 15:50:08.929058,B1.2,24,5PM,GT
9,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,5,Survival,Practice,0.0,<NA>,2026-09-22 15:50:08.929058,B1.2,24,5PM,GT


## Carga a SQL Server

Servidor `CCA-CENT-R5186`, base `rendimiento_academico`, tabla `dbo.avance_oxford`, con la
sesión de Windows. `id` no se envía: lo genera la tabla.

Corre las celdas 12 a 17 en orden. La 15 es la única que escribe en la base.

In [13]:
# 12. ¿Estan las librerias para hablar con SQL Server?
try:
    import pyodbc
    import sqlalchemy

    print("sqlalchemy", sqlalchemy.__version__, "| pyodbc", pyodbc.version)
    print("drivers instalados:", [d for d in pyodbc.drivers() if "SQL Server" in d])
except ModuleNotFoundError as falta:
    print("Falta instalar:", falta.name)
    print("Corre esto en una celda y vuelve a ejecutar esta:")
    print("    %pip install sqlalchemy pyodbc")

sqlalchemy 2.0.54 | pyodbc 5.3.0
drivers instalados: ['SQL Server', 'ODBC Driver 18 for SQL Server', 'ODBC Driver 17 for SQL Server']


In [14]:
# 13. Conexion (autenticacion de Windows, no se escriben usuario ni clave)
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

SERVIDOR = "CCA-CENT-R5186"          # instancia con nombre: "CCA-CENT-R5186\\SQLEXPRESS"
BASE = "rendimiento_academico"
TABLA = "avance_oxford"

# De mejor a peor: el driver viejo ("SQL Server") no sabe mandar lotes ni columnas
# que vengan completas en NULL, por eso se prefiere cualquier "ODBC Driver".
PREFERIDOS = ["ODBC Driver 18 for SQL Server", "ODBC Driver 17 for SQL Server",
              "ODBC Driver 13 for SQL Server", "SQL Server Native Client 11.0", "SQL Server"]
instalados = pyodbc.drivers()
driver = next((d for d in PREFERIDOS if d in instalados), None)
if driver is None:
    raise RuntimeError(f"No hay driver de SQL Server. Instalados: {instalados}")

MODERNO = driver.startswith("ODBC Driver")
cadena = (f"DRIVER={{{driver}}};SERVER={SERVIDOR};DATABASE={BASE};"
          "Trusted_Connection=yes;TrustServerCertificate=yes")
motor = create_engine(URL.create("mssql+pyodbc", query={"odbc_connect": cadena}),
                      fast_executemany=MODERNO)

with motor.connect() as conexion:
    quien = conexion.execute(text("SELECT SUSER_NAME(), DB_NAME()")).one()
    antes = conexion.execute(text(f"SELECT COUNT(*) FROM dbo.{TABLA}")).scalar()
print(f"driver elegido: {driver}" + ("" if MODERNO else "   <- viejo: la carga ira mas lenta"))
print(f"drivers instalados: {instalados}")
print(f"conectado como {quien[0]} a {quien[1]}")
print(f"la tabla tiene hoy {antes:,} filas")

driver elegido: ODBC Driver 18 for SQL Server
drivers instalados: ['SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)', 'ODBC Driver 18 for SQL Server', 'ODBC Driver 17 for SQL Server']
conectado como COLOMBO\sebastian.mendez a rendimiento_academico
la tabla tiene hoy 3,425 filas


In [15]:
# 14. Dejar el dataset listo para la tabla (NaN/NaT -> NULL, fecha sin hora)
from sqlalchemy.types import Date, DateTime, Float, Integer, NVARCHAR

TIPOS = {
    "teacher": NVARCHAR(200), "learning_material": NVARCHAR(200), "student": NVARCHAR(200),
    "last_opened": Date(), "unit_number": Integer(), "unit_title": NVARCHAR(200),
    "item_type": NVARCHAR(50), "completed_pct": Float(), "score_pct": Float(),
    "created_at": DateTime(),
    "clase": NVARCHAR(30), "codigo": NVARCHAR(30),      # columnas agregadas a la tabla
    "grupo": NVARCHAR(30), "segmento": NVARCHAR(30),
}

para_sql = avance_oxford.copy()
para_sql["last_opened"] = para_sql["last_opened"].dt.date     # en SQL la columna es DATE
para_sql = para_sql.astype(object).where(para_sql.notna(), None)

faltan = [c for c in TIPOS if c not in para_sql.columns]
sobran = [c for c in para_sql.columns if c not in TIPOS]
print("columnas que faltan:", faltan or "ninguna", "| columnas de mas:", sobran or "ninguna")

repetidas = para_sql.duplicated(subset=["learning_material", "student", "clase", "codigo",
                                        "unit_number", "unit_title", "item_type"]).sum()
print(f"{len(para_sql):,} filas listas para enviar")
print(f"filas repetidas dentro del propio dataset: {repetidas}")
print("cursos que se van a cargar:")
display(para_sql.groupby(["clase", "grupo", "codigo", "segmento"], dropna=False).size()
        .rename("filas").reset_index())
para_sql.head(3)

columnas que faltan: ninguna | columnas de mas: ninguna
3,425 filas listas para enviar
filas repetidas dentro del propio dataset: 0
cursos que se van a cargar:


,clase,grupo,codigo,segmento,filas
0,B1.2,5PM,22,BPO,575
1,B1.2,5PM,23,BPO,575
2,B1.2,5PM,24,GT,450
3,B1.2,5PM,26,TIC,700
4,B1.2,5PM,27,TIC,775
5,B1.2,5PM,28,IAV,350


,teacher,learning_material,student,last_opened,unit_number,unit_title,item_type,completed_pct,score_pct,created_at,clase,codigo,grupo,segmento
0,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,None,Total / General,Overall,0.0,None,2026-09-22 15:50:08.929058,B1.2,24,5PM,GT
1,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,1,Achievements,Practice,0.0,None,2026-09-22 15:50:08.929058,B1.2,24,5PM,GT
2,Daniel Alvarez,Wide Angle Level 4 Online Practice,Armando Pacheco,2025-12-03,1,Test,Test,0.0,None,2026-09-22 15:50:08.929058,B1.2,24,5PM,GT


In [16]:
# 15. Preparar la tabla para actualizar en vez de duplicar.
#     Se puede correr las veces que quieras: si ya esta hecho, no hace nada.
STAGE = f"{TABLA}_stage"        # tabla de paso: recibe la descarga de hoy

# La llave natural: lo que hace unica a una fila. Tiene que ser IGUAL en las dos tablas.
LLAVE = """CONCAT(learning_material, '|', student, '|', clase, '|', grupo, '|', codigo, '|',
                  ISNULL(segmento, ''), '|', ISNULL(unit_number, -1), '|', unit_title, '|',
                  item_type)"""

COLUMNAS = """teacher, learning_material, student, last_opened, unit_number, unit_title,
              item_type, completed_pct, score_pct, created_at, clase, codigo, grupo, segmento"""

PASOS = [
    # Cuando cambio de valor por ultima vez (created_at se queda como "cuando aparecio")
    f"""IF COL_LENGTH('dbo.{TABLA}', 'updated_at') IS NULL
            ALTER TABLE dbo.{TABLA} ADD updated_at DATETIME2 NULL;""",
    # Huella de la llave: 32 bytes en vez de comparar 9 columnas
    f"""IF COL_LENGTH('dbo.{TABLA}', 'clave_hash') IS NULL
            ALTER TABLE dbo.{TABLA} ADD clave_hash
                AS CONVERT(BINARY(32), HASHBYTES('SHA2_256', {LLAVE})) PERSISTED;""",
    # La tabla de paso se crea con los MISMOS tipos, si no el hash no coincidiria
    f"""IF OBJECT_ID('dbo.{STAGE}') IS NULL
            EXEC('SELECT TOP 0 {COLUMNAS} INTO dbo.{STAGE} FROM dbo.{TABLA}');""",
    f"""IF COL_LENGTH('dbo.{STAGE}', 'clave_hash') IS NULL
            ALTER TABLE dbo.{STAGE} ADD clave_hash
                AS CONVERT(BINARY(32), HASHBYTES('SHA2_256', {LLAVE})) PERSISTED;""",
]

with motor.begin() as conexion:
    for paso in PASOS:
        conexion.exec_driver_sql(paso)

    # El indice unico solo se puede crear si hoy no hay duplicados
    repetidas_en_tabla = conexion.exec_driver_sql(
        f"SELECT COUNT(*) FROM (SELECT clave_hash FROM dbo.{TABLA} "
        "GROUP BY clave_hash HAVING COUNT(*) > 1) AS x").scalar()
    hay_indice = conexion.exec_driver_sql(
        "SELECT COUNT(*) FROM sys.indexes WHERE name = 'UX_avance_oxford_clave' "
        f"AND object_id = OBJECT_ID('dbo.{TABLA}')").scalar()

    if hay_indice:
        print("El indice unico ya existe: no se puede duplicar una fila.")
    elif repetidas_en_tabla:
        print(f"OJO: hay {repetidas_en_tabla:,} llaves repetidas de cargas anteriores.")
        print("Limpia la tabla (o borra los duplicados) y vuelve a correr esta celda:")
        print(f"    DELETE FROM dbo.{TABLA};")
    else:
        conexion.exec_driver_sql(
            f"CREATE UNIQUE INDEX UX_avance_oxford_clave ON dbo.{TABLA} (clave_hash)")
        print("Indice unico creado: desde ahora es imposible duplicar una fila.")

print(f"tabla de paso lista: dbo.{STAGE}")


El indice unico ya existe: no se puede duplicar una fila.
tabla de paso lista: dbo.avance_oxford_stage


In [17]:
# 16. La carga: actualiza lo que ya existe e inserta lo nuevo (esta celda escribe en la base)
import time


def cargar_a_mano(df, tabla):
    """Respaldo para drivers viejos: lotes sin adivinar tipos.
    Las fechas van como texto ISO, que SQL Server convierte solo."""
    datos = df.copy()
    datos["last_opened"] = datos["last_opened"].map(lambda f: f.isoformat() if f else None)
    datos["created_at"] = datos["created_at"].map(
        lambda f: f.strftime("%Y-%m-%d %H:%M:%S.%f")[:-3] if f else None)
    sql = (f"INSERT INTO dbo.{tabla} ({', '.join(datos.columns)}) "
           f"VALUES ({', '.join(['?'] * len(datos.columns))})")
    filas = list(datos.itertuples(index=False, name=None))
    conexion = pyodbc.connect(cadena)          # sin commit no queda nada, si algo falla
    try:
        cursor = conexion.cursor()
        for desde in range(0, len(filas), 500):
            cursor.executemany(sql, filas[desde:desde + 500])
        conexion.commit()
    finally:
        conexion.close()


# Solo se actualiza si de verdad cambio algo: asi updated_at significa "cambio ese dia".
# El EXCEPT compara bien los NULL, cosa que un <> no hace.
SQL_ACTUALIZAR = f"""
UPDATE destino
   SET destino.completed_pct = origen.completed_pct,
       destino.score_pct     = origen.score_pct,
       destino.last_opened   = origen.last_opened,
       destino.teacher       = origen.teacher,
       destino.updated_at    = SYSDATETIME()
  FROM dbo.{TABLA} AS destino
  JOIN dbo.{STAGE} AS origen ON origen.clave_hash = destino.clave_hash
 WHERE EXISTS (SELECT origen.completed_pct, origen.score_pct, origen.last_opened, origen.teacher
               EXCEPT
               SELECT destino.completed_pct, destino.score_pct, destino.last_opened,
                      destino.teacher);
"""

SQL_INSERTAR = f"""
INSERT INTO dbo.{TABLA} (teacher, learning_material, student, last_opened, unit_number,
                         unit_title, item_type, completed_pct, score_pct, created_at,
                         clase, codigo, grupo, segmento)
SELECT origen.teacher, origen.learning_material, origen.student, origen.last_opened,
       origen.unit_number, origen.unit_title, origen.item_type, origen.completed_pct,
       origen.score_pct, origen.created_at, origen.clase, origen.codigo, origen.grupo,
       origen.segmento
  FROM dbo.{STAGE} AS origen
 WHERE NOT EXISTS (SELECT 1 FROM dbo.{TABLA} AS d WHERE d.clave_hash = origen.clave_hash);
"""

inicio = time.time()

# 1. La descarga de hoy entra a la tabla de paso (se vacia primero)
try:
    with motor.begin() as conexion:
        conexion.exec_driver_sql(f"TRUNCATE TABLE dbo.{STAGE}")
        para_sql.to_sql(STAGE, conexion, schema="dbo", if_exists="append", index=False,
                        chunksize=1000 if MODERNO else 200, dtype=TIPOS)
except Exception as error:
    print("to_sql no pudo:", str(error).splitlines()[0])
    print("Reintento con el metodo compatible con drivers viejos...")
    with motor.begin() as conexion:
        conexion.exec_driver_sql(f"TRUNCATE TABLE dbo.{STAGE}")
    cargar_a_mano(para_sql, STAGE)

# 2. De la tabla de paso a la definitiva, todo o nada
with motor.begin() as conexion:
    repetidas_hoy = conexion.exec_driver_sql(
        f"SELECT COUNT(*) FROM (SELECT clave_hash FROM dbo.{STAGE} "
        "GROUP BY clave_hash HAVING COUNT(*) > 1) AS x").scalar()
    if repetidas_hoy:
        raise ValueError(f"La descarga de hoy trae {repetidas_hoy:,} llaves repetidas "
                         "(¿el mismo curso en dos archivos?). No se aplico nada.")
    actualizadas = conexion.exec_driver_sql(SQL_ACTUALIZAR).rowcount
    insertadas = conexion.exec_driver_sql(SQL_INSERTAR).rowcount

print(f"{len(para_sql):,} filas procesadas en {time.time() - inicio:.1f} s")
print(f"  actualizadas (cambio su valor): {actualizadas:,}")
print(f"  insertadas (nuevas):            {insertadas:,}")
print(f"  sin cambios:                    {len(para_sql) - actualizadas - insertadas:,}")


3,425 filas procesadas en 0.5 s
  actualizadas (cambio su valor): 0
  insertadas (nuevas):            0
  sin cambios:                    3,425


In [18]:
# 17. Comprobar contra la base
with motor.connect() as conexion:
    despues = conexion.execute(text(f"SELECT COUNT(*) FROM dbo.{TABLA}")).scalar()
print(f"antes: {antes:,}  ->  despues: {despues:,}   (filas nuevas: {despues - antes:,})")

# Lo que se movio hoy
cambios = pd.read_sql(text(f"""
    SELECT CAST(updated_at AS DATE) AS dia, COUNT(*) AS filas_cambiadas
    FROM dbo.{TABLA}
    WHERE updated_at IS NOT NULL
    GROUP BY CAST(updated_at AS DATE)
    ORDER BY dia DESC
"""), motor)
print("\nCambios por dia:")
display(cambios)

# El avance por unidad, que es para lo que sirve todo esto
avance_por_unidad = pd.read_sql(text(f"""
    SELECT clase, grupo, codigo, segmento, unit_number, unit_title,
           COUNT(DISTINCT student) AS estudiantes,
           ROUND(AVG(completed_pct), 1) AS avance_promedio,
           ROUND(AVG(score_pct), 1) AS nota_promedio
    FROM dbo.{TABLA}
    WHERE item_type <> 'Overall'
    GROUP BY clase, grupo, codigo, segmento, unit_number, unit_title
    ORDER BY clase, grupo, codigo, unit_number
"""), motor)
display(avance_por_unidad)

pd.read_sql(text(f"SELECT TOP 10 * FROM dbo.{TABLA} ORDER BY id DESC"), motor)


antes: 3,425  ->  despues: 3,425   (filas nuevas: 0)

Cambios por dia:


,dia,filas_cambiadas


,clase,grupo,codigo,segmento,unit_number,unit_title,estudiantes,avance_promedio,nota_promedio
0,B1.2,5PM,22,BPO,1,Achievements,23,9.3,61.0
1,B1.2,5PM,22,BPO,1,Test,23,8.7,100.0
2,B1.2,5PM,22,BPO,2,News,23,8.7,88.4
3,B1.2,5PM,22,BPO,2,Test,23,8.7,98.1
4,B1.2,5PM,22,BPO,3,Frontiers,23,8.7,92.0
...,...,...,...,...,...,...,...,...,...
139,B1.2,5PM,28,IAV,10,Test,14,2.4,21.4
140,B1.2,5PM,28,IAV,11,Test,14,2.4,33.3
141,B1.2,5PM,28,IAV,11,Wishes,14,0.4,5.8
142,B1.2,5PM,28,IAV,12,Test,14,2.4,28.6


,id,teacher,learning_material,student,last_opened,unit_number,unit_title,item_type,completed_pct,score_pct,created_at,clase,codigo,grupo,segmento,updated_at,clave_hash
0,7795,Daniel Alvarez,Wide Angle Level 4 Online Practice,Johanna Maribel Pedroza Perez,None,12,Test,Test,0.0,None,2026-09-22 12:08:38.837,B1.2,28,5PM,IAV,None,b'z\xeb\xa5s\xcfe\xfeF\x93\xba\xedJ=g#\xdf\xa7...
1,7794,Daniel Alvarez,Wide Angle Level 4 Online Practice,Johanna Maribel Pedroza Perez,None,12,Wisdom,Practice,0.0,None,2026-09-22 12:08:38.837,B1.2,28,5PM,IAV,None,"b'\xf9:it\xca,vt\xae\x93\xfd0R\x83-\x98\x01z\x..."
2,7793,Daniel Alvarez,Wide Angle Level 4 Online Practice,Johanna Maribel Pedroza Perez,None,11,Test,Test,0.0,None,2026-09-22 12:08:38.837,B1.2,28,5PM,IAV,None,"b""\x8c\xf2\xe9\xf6\xff\x15t[\xee\xf7'Z2\x08S\x..."
3,7792,Daniel Alvarez,Wide Angle Level 4 Online Practice,Johanna Maribel Pedroza Perez,None,11,Wishes,Practice,0.0,None,2026-09-22 12:08:38.837,B1.2,28,5PM,IAV,None,b'\xa4\xf7\x91\xe7r+D\xd6a\xb1\xb3\x82\xc3\x07...
4,7791,Daniel Alvarez,Wide Angle Level 4 Online Practice,Johanna Maribel Pedroza Perez,None,10,Test,Test,0.0,None,2026-09-22 12:08:38.837,B1.2,28,5PM,IAV,None,b'\x91f6\xf5\xc5+\xa8\x0f\\J\xabo\xc5Y\x04\xc6...
5,7790,Daniel Alvarez,Wide Angle Level 4 Online Practice,Johanna Maribel Pedroza Perez,None,10,Mysteries,Practice,0.0,None,2026-09-22 12:08:38.837,B1.2,28,5PM,IAV,None,"b'\x12PQs\xbc\xee*\xc8\xe26""L\x81\xc1I\xd3\x04..."
6,7789,Daniel Alvarez,Wide Angle Level 4 Online Practice,Johanna Maribel Pedroza Perez,None,9,Test,Test,0.0,None,2026-09-22 12:08:38.837,B1.2,28,5PM,IAV,None,b'{\x87\xe0\xb9~P\x01y\x11\xbc\xb9\xda&\x06\xc...
7,7788,Daniel Alvarez,Wide Angle Level 4 Online Practice,Johanna Maribel Pedroza Perez,None,9,Reviews,Practice,0.0,None,2026-09-22 12:08:38.837,B1.2,28,5PM,IAV,None,b'd\xb4\x88t7\x01\x0b{\xe15Jnp\x0fr\xd6e\x87\x...
8,7787,Daniel Alvarez,Wide Angle Level 4 Online Practice,Johanna Maribel Pedroza Perez,None,8,Test,Test,0.0,None,2026-09-22 12:08:38.837,B1.2,28,5PM,IAV,None,b'\x11\x07(\xa3\xbd8C\xe7\xee\x9e\x91=@\x1c\x8...
9,7786,Daniel Alvarez,Wide Angle Level 4 Online Practice,Johanna Maribel Pedroza Perez,None,8,Change,Practice,0.0,None,2026-09-22 12:08:38.837,B1.2,28,5PM,IAV,None,b'OT\xadR.E}\x0c\xcd\xfd\x0b\xc1\xfaL4#_q\xb3\...
